<a href="https://colab.research.google.com/github/133Haseeb/MLflyrankhaseeb/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/133Haseeb/MLflyrankhaseeb/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*



One row represents the daily search performance of one content page for one client on one report date.

For this assignment, I will work with the Search Intelligence warehouse data and use a mid-panel month to avoid leakage. The unit of analysis will be verified with SQL queries.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

## Fields

### Features
- gsc_impressions
- gsc_clicks
- gsc_avg_position
- report_date
- content_hash_id

### Label
- Content decline (derived later from impression changes)

### Context
- client_hash_id
- report_date

### Excluded
I exclude any label-derived information or future outcome variables because they would cause data leakage and produce unrealistic model performance.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [2]:
%pip -q install duckdb huggingface_hub

In [3]:
import os
from google.colab import userdata

HF_TOKEN = userdata.get("HF_Token")

In [4]:
import duckdb

con = duckdb.connect()

con.execute(f"""
CREATE OR REPLACE SECRET hf
(TYPE huggingface, TOKEN '{HF_TOKEN}')
""")

REL = "hf://datasets/FlyRank/internship-warehouse"

TABLES = {
    "fact_daily": f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    "dim_clients": f"read_parquet('{REL}/dim_clients.parquet')",
    "dim_content": f"read_parquet('{REL}/dim_content.parquet')",
    "fact_query_90d": f"read_parquet('{REL}/fact_content_query_90d.parquet')"
}

print("Connected Successfully!")

Connected Successfully!


In [5]:
grain = con.sql(f"""
SELECT *
FROM {TABLES['fact_daily']}
LIMIT 5
""").df()

grain

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2025-01-27,client_9958f0a7ae1df715,content_3b70a18ea133b2bb,True,True,True,False,30,0,115,...,0,0,0,0,0,0,0,0,0,2025-01
1,2025-01-27,client_9958f0a7ae1df715,content_fe8e8155ce1d47a2,True,True,True,False,5,0,358,...,0,0,0,0,0,0,0,0,0,2025-01
2,2025-01-27,client_9958f0a7ae1df715,content_b4462a1b90640058,True,True,True,False,1,0,34,...,0,0,0,0,0,0,0,0,0,2025-01
3,2025-01-27,client_9958f0a7ae1df715,content_c899aef92518c714,True,True,True,False,6,0,140,...,0,0,0,0,0,0,0,0,0,2025-01
4,2025-01-27,client_9958f0a7ae1df715,content_c7c1d2e68d9d0964,True,True,True,False,5,0,89,...,0,0,0,0,0,0,0,0,0,2025-01


In [6]:
rows = con.sql(f"""
SELECT COUNT(*) AS total_rows
FROM {TABLES['fact_daily']}
""").df()

rows

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows
0,78835655


In [7]:
dates = con.sql(f"""
SELECT
MIN(report_date) AS start_date,
MAX(report_date) AS end_date
FROM {TABLES['fact_daily']}
""").df()

dates

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,start_date,end_date
0,2025-01-27,2026-06-30


In [9]:
columns = con.sql(f"""
DESCRIBE
SELECT *
FROM {TABLES['fact_daily']}
""").df()

columns

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


In [10]:
availability = con.sql(f"""
SELECT
COUNT(*) AS available_rows
FROM {TABLES['fact_daily']}
WHERE month='2026-03'
AND gsc_data_available IS TRUE
""").df()

availability

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,available_rows
0,3611061


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

## Data Limits

This dataset contains anonymized search performance information and cannot explain why Google rankings changed.

Some clients have more historical data than others, making the dataset unbalanced.

The data should be used for decision support rather than proving cause-and-effect relationships.

The model cannot predict Google's algorithm; it can only identify useful patterns from historical observations.

## Self-check

Before you submit, confirm each line honestly:

- [ Checked] Every section above is filled — markdown thinking AND the code that backs it
- [ checked] The notebook runs top to bottom with no errors (Runtime → Run all)
- [checked ] No client names, URLs, or private queries anywhere
- [checked ] My claims use careful words: observed, measured, directional, decision-support
- [checked ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.